Fine-Tuning BERT on a Kaggle Dataset

In [6]:
# install libraries
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [33]:
# load dataset
df = pd.read_csv(
    "IMDB Dataset.csv",
    engine='python',
    encoding='utf-8',
    on_bad_lines='skip'
)
print(df.shape)
df.head()

(769, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [34]:
# Step 1 Data preprocesing
# Check missing values
print(df.isnull().sum())

# Drop missing rows (if any)
df = df.dropna()

# Convert labels to numbers
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

# Basic text cleaning
df['review'] = df['review'].str.lower()
df['review'] = df['review'].str.replace('<br />', '', regex=False)

df.head()

review       0
sentiment    0
dtype: int64


,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. the filming tec...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1


In [35]:
#Step 2 Data splitting
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.3, random_state=42)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42)
print(df.head())

                                              review  sentiment
0  one of the other reviewers has mentioned that ...          1
1  a wonderful little production. the filming tec...          1
2  i thought this was a wonderful way to spend ti...          1
3  basically there's a family where a little boy ...          0
4  petter mattei's "love in the time of money" is...          1


In [36]:
# Remove NaN from train
train_data = [(t, l) for t, l in zip(train_texts, train_labels) if str(l) != 'nan']
train_texts, train_labels = zip(*train_data)

# Remove NaN from val
val_data = [(t, l) for t, l in zip(val_texts, val_labels) if str(l) != 'nan']
val_texts, val_labels = zip(*val_data)

# Convert to list again
train_texts, train_labels = list(train_texts), [int(x) for x in train_labels]
val_texts, val_labels = list(val_texts), [int(x) for x in val_labels]


In [37]:
# Step 3 Tokenization 
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Convert text into tokens
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True
)
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)
import torch
from torch.utils.data import Dataset

class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

In [38]:
# Step 4: Model Building (BERT)
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
model.config.problem_type = "single_label_classification"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
# Step 5: Fine-Tuning BERT
# Optimizer → AdamW
# Learning rate → 2e-5
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW  
from tqdm import tqdm


# ---------------- Dataset Class ----------------
class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

# ---------------- Create Dataset ----------------
train_labels = [int(x) for x in train_labels]
val_labels = [int(x) for x in val_labels]
train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training
epochs = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} completed | Total Loss: {total_loss}")

100%|██████████████████████████████████████████████████████████████████████████████████| 68/68 [11:50<00:00, 10.44s/it]


Epoch 1 completed | Total Loss: 40.70420069992542


100%|██████████████████████████████████████████████████████████████████████████████████| 68/68 [12:32<00:00, 11.06s/it]

Epoch 2 completed | Total Loss: 16.804560035467148


In [40]:
 # step 6Evaluation (Accuracy, F1, etc.)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# Metrics
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.8521739130434782
Precision: 0.851063829787234
Recall: 0.8
F1 Score: 0.8247422680412371


In [1]:
# EXPERIMENT 1
import torch
from transformers import AutoModelForSequenceClassification

#  Define device FIRST
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Move model to device
model.to(device)

# Freeze BERT layers
for param in model.bert.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# Expermient 2
import torch
from transformers import AutoModelForSequenceClassification

# ✅ Device
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# ✅ Load fresh model (VERY IMPORTANT)
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.to(device)

# ✅ Step 1: Freeze ALL layers
for param in model.bert.parameters():
    param.requires_grad = False

# ✅ Step 2: Unfreeze LAST 2 layers
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

# Classifier is already trainable by default

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
